In [ ]:
# Colab setup - run this first, then switch the runtime to R
# (Runtime > Change runtime type > R). Precompiled Linux binaries via Posit P3M
# keep the install to a couple of minutes instead of compiling from source.
local({
  cn <- tryCatch(system("lsb_release -cs", intern = TRUE), error = function(e) "jammy")
  options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", cn)),
          HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
})
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(
  c("SingleCellExperiment", "SummarizedExperiment", "MultiAssayExperiment",
    "S4Vectors", "basilisk"),
  update = FALSE, ask = FALSE)
install.packages(c("remotes", "reticulate", "Matrix", "ggplot2"))
remotes::install_github("PYangLab/M3", subdir = "m3-r")
# The first m3_train() builds the bundled Python engine via basilisk (a few
# minutes on first run); afterwards it is cached for the session.

# Feature attribution

m3 explains its donor-level disease prediction with **end-to-end integrated gradients**, attributing the prediction back to **genes/proteins**, **cell types**, and **donors**. We train on the full demo reference, then call `m3_attribute(reference_labels = "HC")` (HC = the healthy integrated-gradients baseline) and visualise the rankings.

## 1. Load the demo dataset

In [ ]:
library(m3)
library(ggplot2)
set.seed(0)
data <- m3_demo()
data

## 2. Train (integration VAE + donor predictor on the full reference)

Attribution runs through the trained `(generator, corrector)`, so we provide `donor_key` + `celltype_key` (no held-out batch — we attribute on the full set). The donor-predictor knobs default to the from-scratch driver values.

In [ ]:
#To save time, users can set max_epochs to 100 for test.
model <- m3_train(
  data,
  condition_keys   = c("cond_group", "Age_interval"),
  target_condition = "cond_group",
  celltype_key     = "mergedcelltype",
  batch_key        = "batch",
  donor_key        = "sample_id",
  embedding_dim    = 30L,
  max_epochs       = 500L,
  seed = 0L
)
m3_capabilities(model)

## 3. Attribute: Severe vs HC baseline

In [ ]:
attr <- m3_attribute(model, reference_labels = "HC")   # n_steps defaults to 50
attr

cat("\ntop cell types (>= 200 cells per condition):\n")

print(m3_top_celltypes(attr, min_cells_per_condition = 200L))

### Per-celltype-balanced gene ranking (the publication recipe)

`attr$genes` is the **raw** ranking — `mean(|IG|)` over all cells. The publication recipe drops cell types with \< 200 cells in either condition, scores each gene by `mean(|gene x celltype IG|)` over the kept cell types, excludes housekeeping / ribosomal genes, and (optionally) restricts to one modality.

In [ ]:
top100_rna <- m3_top_genes(attr, n = 100L, min_cells_per_condition = 200L, modality = "rna")
cat(sprintf("top-100 RNA genes (computed over %d balanced cell types):\n",
            top100_rna$n_celltypes_used[1]))

print(utils::head(top100_rna, 15))

## 4. Visualise — top genes, top cell types, gene x celltype heatmap

In [ ]:
g <- utils::head(top100_rna, 20)
p1 <- ggplot(g, aes(stats::reorder(feature, score), score)) +
  geom_col(fill = "#4c72b0") + coord_flip() +
  labs(title = "Top-20 RNA genes (per-celltype-balanced, housekeeping excluded)",
       x = NULL, y = "mean |IG| across balanced cell types") +
  theme_classic() + theme(axis.text.y = element_text(size = 7))

c20 <- m3_top_celltypes(attr, min_cells_per_condition = 200L)
p2 <- ggplot(c20, aes(stats::reorder(celltype, importance), importance)) +
  geom_col(fill = "#dd8452") + coord_flip() +
  labs(title = "Cell-type importance (>= 200 cells per condition)", x = NULL, y = "importance") +
  theme_classic() + theme(axis.text.y = element_text(size = 8))
print(p1); print(p2)

### Gene x cell-type attribution heatmap (top-30 RNA genes)

In [ ]:
gcm <- m3_gene_celltype_matrix(attr)                 # celltype x feature
top_features <- utils::head(top100_rna$feature, 30)
idx <- match(top_features, attr$feature_names)
sub <- gcm[, idx, drop = FALSE]
hm <- expand.grid(celltype = attr$celltype_names, feature = top_features,
                  stringsAsFactors = FALSE)
hm$value <- as.vector(sub)
hm$feature <- factor(hm$feature, levels = top_features)
hm$celltype <- factor(hm$celltype, levels = attr$celltype_names)
lim <- max(abs(hm$value), na.rm = TRUE)
ggplot(hm, aes(feature, celltype, fill = value)) +
  geom_tile() +
  scale_fill_gradient2(low = "#053061", mid = "white", high = "#67001f",
                       midpoint = 0, limits = c(-lim, lim), name = "signed IG") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 6),
        axis.text.y = element_text(size = 7)) +
  labs(title = "Gene x cell-type attribution (top-30 RNA genes, balanced ranking)",
       x = NULL, y = NULL)

**Done.** End-to-end integrated-gradients attribution: ranked gene/protein, cell-type and donor importance for the Severe-vs-HC prediction.